# Session Selection — DANDI_000688

Scan all DANDI_000688 sessions (2013-2016) in the local zarr store.
Extract metrics: electrode count, trial count, duration, firing rate, velocity symmetry.
Filter by quality criteria and rank candidates for analysis.

## Candidate Selection Criteria
- **n_trials per session ≥ 120** — enough trials for robust K-Fold CV
- **n_active_electrodes ≥ 150** — sufficient neural signal
- **symmetry_ratio ≥ 0.7** — vx and vy have comparable variability (min/max std ratio)

In [28]:
import numpy as np
import pandas as pd
import zarr
from pathlib import Path

ZARR_PATH = "/home/camilavelasquez/Documents/Datasets/Combined_Motor_Datasets_V9.zarr"
BIN_SIZE_MS = 50

print(f"Opening zarr store: {ZARR_PATH}")

Opening zarr store: /home/camilavelasquez/Documents/Datasets/Combined_Motor_Datasets_V9.zarr


## List All Sessions

In [29]:
store = zarr.DirectoryStore(ZARR_PATH)
root = zarr.group(store=store, overwrite=False)

all_sessions = list(root.keys())
print(f"Total sessions in store: {len(all_sessions)}")
print(f"Sample keys: {all_sessions[:5]}")

Total sessions in store: 168
Sample keys: ['20090812', '20090819', '20090910', '20090912', '20090916']


## Filter to DANDI_000688 Years (2013-2016)

In [30]:
# DANDI_000688 has exactly 111 known sessions (subject C, center-out task)
DANDI_000688_sessions = [
    '20130819','20130820','20130821','20130822','20130823','20130830',
    '20130903','20130904','20130905','20130906','20130909','20130910',
    '20131003','20131009','20131010','20131011','20131022','20131023',
    '20131028','20131029','20131031','20131101','20131203','20131204',
    '20131209','20131210','20131212','20131213','20131217','20131218',
    '20131219','20131220','20140114','20140115','20140116','20140203',
    '20140214','20140217','20140218','20140221','20140224','20140303',
    '20140304','20140306','20140307','20140626','20140627','20140929',
    '20141203','20150309','20150311','20150312','20150313','20150316',
    '20150317','20150318','20150319','20150320','20150511','20150512',
    '20150610','20150611','20150612','20150615','20150616','20150617',
    '20150623','20150625','20150626','20150629','20150630','20150701',
    '20150703','20150706','20150707','20150708','20150709','20150710',
    '20150713','20150714','20150715','20150716','20151103','20151104',
    '20151106','20151109','20151110','20151112','20151113','20151116',
    '20151117','20151119','20151120','20151201','20160405','20160406',
    '20160407','20160909','20160912','20160914','20160915','20160919',
    '20160921','20160923','20160929','20161005','20161006','20161007',
    '20161011','20161013','20161021'
]

# Filter to only known DANDI_000688 sessions that exist in the store
dandi_sessions = [s for s in DANDI_000688_sessions if s in all_sessions]

assert len(DANDI_000688_sessions) == 111, f"Expected 111 DANDI_000688 sessions, got {len(DANDI_000688_sessions)}"
print(f"DANDI_000688 known sessions: {len(DANDI_000688_sessions)} (expected 111) ✓")
print(f"Sessions present in store: {len(dandi_sessions)}")
if len(dandi_sessions) < len(DANDI_000688_sessions):
    missing = set(DANDI_000688_sessions) - set(dandi_sessions)
    print(f"Missing from store ({len(missing)}): {sorted(missing)[:10]}...")

DANDI_000688 known sessions: 111 (expected 111) ✓
Sessions present in store: 111


## Extract Metrics for Each Session

For each session, compute:
- **n_electrodes**: number of spike channels
- **n_bins**: total time bins (already 50 ms binned in zarr)
- **duration_min**: session duration in minutes
- **n_trials**: unique positive trial IDs (excluding 0)
- **n_active_electrodes**: channels with mean FR > 0.5 Hz
- **mean_fr**: mean firing rate across all channels (Hz)
- **vx_std, vy_std**: velocity component std (velocity already shape (n_bins, 2))
- **symmetry_ratio**: min/max of (vx_std, vy_std) — closer to 1.0 is better

Note: Data stored under 'processed data' key. Velocity is pre-shaped as (n_bins, 2).

In [31]:
results = []

for session_key in dandi_sessions:
    try:
        # Data is stored under 'processed data' key
        g = root[session_key]['processed data']
        
        # Extract data
        spikes = g['spikes'][:]  # shape (n_electrodes, n_bins)
        velocity = g['velocity'][:]  # shape (n_bins, 2)
        trial_ids = g['trial_id'][:]  # shape (n_bins,)
        
        n_electrodes = spikes.shape[0]
        n_bins = spikes.shape[1]
        duration_min = n_bins * BIN_SIZE_MS / 60000.0  # convert to minutes
        
        # Compute binned spike counts (already binned in zarr)
        mean_fr_per_el = spikes.mean() / (BIN_SIZE_MS / 1000.0)  # Hz
        active_els = (spikes.mean(axis=1) / (BIN_SIZE_MS / 1000.0) > 0.5).sum()
        
        # Count unique positive trial IDs
        unique_trials = np.unique(trial_ids[trial_ids > 0])
        n_trials = len(unique_trials)
        
        # Velocity: already shape (n_bins, 2), no transpose needed
        vx_std = velocity[:, 0].std()
        vy_std = velocity[:, 1].std()
        symmetry = min(vx_std, vy_std) / max(vx_std, vy_std) if max(vx_std, vy_std) > 0 else 0.0
        
        results.append({
            'session': session_key,
            'n_electrodes': n_electrodes,
            'n_bins': n_bins,
            'duration_min': round(duration_min, 2),
            'n_trials': n_trials,
            'n_active_electrodes': int(active_els),
            'mean_fr': round(mean_fr_per_el, 2),
            'vx_std': round(vx_std, 4),
            'vy_std': round(vy_std, 4),
            'symmetry_ratio': round(symmetry, 3),
        })
        print(f'✓ {session_key}: {n_trials} trials, {n_electrodes} els, {int(active_els)} active, sym={symmetry:.3f}')
    except Exception as e:
        print(f'✗ {session_key}: {e}')
        continue

print(f"\nSuccessfully processed {len(results)} sessions")

✓ 20130819: 143 trials, 54 els, 0 active, sym=0.926
✓ 20130820: 125 trials, 53 els, 0 active, sym=0.979
✓ 20130821: 164 trials, 58 els, 1 active, sym=0.989
✓ 20130822: 192 trials, 63 els, 1 active, sym=0.993
✓ 20130823: 213 trials, 44 els, 0 active, sym=0.921
✓ 20130830: 145 trials, 35 els, 0 active, sym=0.955
✓ 20130903: 167 trials, 52 els, 1 active, sym=0.988
✓ 20130904: 116 trials, 72 els, 1 active, sym=0.689
✓ 20130905: 175 trials, 64 els, 0 active, sym=0.947
✓ 20130906: 178 trials, 52 els, 1 active, sym=0.811
✓ 20130909: 218 trials, 58 els, 0 active, sym=0.962
✓ 20130910: 174 trials, 68 els, 1 active, sym=0.833
✓ 20131003: 159 trials, 70 els, 3 active, sym=0.827
✓ 20131009: 136 trials, 73 els, 2 active, sym=0.800
✓ 20131010: 149 trials, 73 els, 1 active, sym=0.749
✓ 20131011: 140 trials, 87 els, 4 active, sym=0.771
✓ 20131022: 152 trials, 40 els, 2 active, sym=0.970
✓ 20131023: 193 trials, 60 els, 5 active, sym=0.990
✓ 20131028: 136 trials, 76 els, 5 active, sym=0.719
✓ 20131029: 

In [35]:
import zarr
print(zarr.__version__)
print(type(store))
# El path está aquí:
if hasattr(store, 'store'):
    print(store.store.path)
elif hasattr(store, '_store'):
    print(store._store.path)
else:
    print(store.chunk_store.path)

2.18.7
<class 'zarr.storage.DirectoryStore'>


AttributeError: 'DirectoryStore' object has no attribute 'chunk_store'

In [37]:
root = zarr.open_group(store, mode='r')
print(list(root.keys())[:5])

['20090812', '20090819', '20090910', '20090912', '20090916']


In [38]:
sess = root['20131203']['processed data']
print("spikes shape:", sess['spikes'].shape)

spikes shape: (47, 1318302)


In [39]:
for sess_id in ['20161021', '20160914', '20160915', '20160919', '20160923']:
    s = root[sess_id]['processed data']
    print(f"{sess_id}: spikes={s['spikes'].shape}, vel={s['velocity'].shape}")

20161021: spikes=(294, 1217250), vel=(1217250, 2)
20160914: spikes=(352, 1072504), vel=(1072504, 2)
20160915: spikes=(308, 956747), vel=(956747, 2)
20160919: spikes=(294, 960302), vel=(960302, 2)
20160923: spikes=(298, 903321), vel=(903321, 2)


In [32]:
def print_tree(group, prefix="", max_depth=4, depth=0):
    if depth >= max_depth:
        return
    for key in list(group.keys())[:10]:  # max 10 por nivel
        item = group[key]
        if hasattr(item, 'shape'):
            print(f"{prefix}{key}: {item.shape} {item.dtype}")
        else:
            print(f"{prefix}{key}/")
            print_tree(item, prefix + "  ", max_depth, depth + 1)

sess = store['20131203']
print_tree(sess)

KeyError: '20131203'

## Session Summary Table (sorted by n_trials)

In [ ]:
df = pd.DataFrame(results)
df = df.sort_values('n_trials', ascending=False).reset_index(drop=True)

print(f"\n=== All {len(df)} Sessions ===")
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', None):
    display(df)

KeyError: 'n_trials'

## Apply Quality Filters

In [ ]:
# Quality thresholds
MIN_TRIALS = 120
MIN_ACTIVE_ELS = 150
MIN_SYMMETRY = 0.7

mask = (df['n_trials'] >= MIN_TRIALS) & \
        (df['n_active_electrodes'] >= MIN_ACTIVE_ELS) & \
        (df['symmetry_ratio'] >= MIN_SYMMETRY)

candidates = df[mask].reset_index(drop=True)

print(f"Sessions passing filters:")
print(f"  n_trials >= {MIN_TRIALS}: {(df['n_trials'] >= MIN_TRIALS).sum()}")
print(f"  n_active_electrodes >= {MIN_ACTIVE_ELS}: {(df['n_active_electrodes'] >= MIN_ACTIVE_ELS).sum()}")
print(f"  symmetry_ratio >= {MIN_SYMMETRY}: {(df['symmetry_ratio'] >= MIN_SYMMETRY).sum()}")
print(f"\nCombined (all 3 criteria): {len(candidates)} sessions\n")

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', None):
    display(candidates)

## Top 5 Candidates (by n_trials)

In [ ]:
top5 = candidates.head(5) if len(candidates) > 0 else df.head(5)

print(f"\n=== Top 5 Sessions (by n_trials) ===")
for idx, row in top5.iterrows():
    print(f"\n[{idx+1}] {row['session']}")
    print(f"    Trials: {row['n_trials']} | Electrodes: {row['n_electrodes']} (active={row['n_active_electrodes']})")
    print(f"    Duration: {row['duration_min']} min | Mean FR: {row['mean_fr']} Hz")
    print(f"    Velocity: vx_std={row['vx_std']}, vy_std={row['vy_std']}, symmetry={row['symmetry_ratio']}")

# Recommendation
if len(candidates) > 0:
    best = candidates.iloc[0]
    print(f"\n→ RECOMMENDED: {best['session']} ({best['n_trials']} trials, {best['n_active_electrodes']} active electrodes)")
else:
    print(f"\n⚠ No sessions pass all quality filters. Relaxing criteria...")
    best = df.iloc[0]
    print(f"→ BEST AVAILABLE: {best['session']} ({best['n_trials']} trials, {best['n_active_electrodes']} active electrodes)")

In [40]:
sess = root['20161021']
print(sess['general']['institution'][:])
print(sess['general']['experimenter'][:])
print(sess['general']['experiment_description'][:])
print(sess['general']['subject']['description'][:])

['Northwestern University']
['Perich, Matthew G.']
['Center-out delayed reaching task']
['Monkey C']
